In [ ]:
 # 1. Uninstall versi lama/rusak
!pip uninstall -y openvino openvino-dev

# 2. Install ulang openvino terbaru yang bersih
!pip install --upgrade --no-cache-dir openvino

Found existing installation: openvino 2026.4.1
Uninstalling openvino-2026.4.1:
  Successfully uninstalled openvino-2026.4.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 36.4 MB/s  0:00:01


In [ ]:
import sys
print("Python version:", sys.version)

try:
    import openvino as ov
    print("✅ OpenVINO berhasil di-load! Versi:", ov.__version__)
except Exception as e:
    print("❌ Masih gagal:", e)

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
✅ OpenVINO berhasil di-load! Versi: 2026.4.1-22982-e213a147257-releases/2026/4


# 🎓 E.S.I.S (Electronic Smoking Identification System)
## Modul Pelatihan FaceNet dengan MTCNN & Optimasi Intel OpenVINO

Buku kerja (notebook) ini dirancang khusus untuk berjalan di **Google Colab** dengan integrasi langsung ke **Google Drive Anda**:
1. **Akses Langsung ke Google Drive**: Memuat dataset mahasiswa langsung dari folder `known_faces` di Drive Anda.
2. **Deteksi & Preprocessing Wajah**: Menggunakan **MTCNN** (Multi-task Cascaded Convolutional Networks).
3. **Ekstraksi Fitur Biometrik**: Menggunakan **FaceNet** (512-dimensi vektor embedding).
4. **Penyimpanan Database Wajah**: Diekspor ke format `.npz` (`data_wajah_facenet_mtcnn.npz`).
5. **Export Model Keras `.h5`**: Menyimpan model FaceNet ke format `.h5` (`facenet_keras.h5`).
6. **Akselerasi Intel OpenVINO**: Mengonversi model ke format OpenVINO IR (`.xml` & `.bin`) agar super cepat di CPU Intel / AMD Server (dilengkapi auto-patch Python 3.13).
7. **Benchmark Kecepatan**: Menguji perbandingan latensi Keras CPU vs OpenVINO CPU (siap dimasukkan ke Bab 4 Skripsi!).
8. **Auto-Backup ke Google Drive & Download Zip**: Otomatis menyimpan hasil model langsung ke folder Drive Anda.


### 📦 Langkah 1: Instalasi Library, Dependensi & Auto-Patch Python 3.13


In [ ]:
# 1. Install Library yang Dibutuhkan
!pip install --quiet keras-facenet mtcnn openvino tqdm scikit-learn

import os
import sys
import time
import shutil
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image as Img
from numpy import asarray, expand_dims
from tqdm import tqdm
from google.colab import files, drive

# Import MTCNN & FaceNet & OpenVINO
from mtcnn import MTCNN
from keras_facenet import FaceNet
import openvino as ov

# --- AUTO-PATCH BUG OPENVINO DI PYTHON 3.13 (PEP 667) ---
# Mengatasi error: NameError: name 'tensorflow' is not defined di utils.py
try:
    ov_dir = os.path.dirname(ov.__file__)
    target_utils = os.path.join(ov_dir, "frontend", "tensorflow", "utils.py")
    if os.path.exists(target_utils):
        with open(target_utils, "r", encoding="utf-8") as f:
            c = f.read()
        if 'exec("del tensorflow")' in c:
            c = c.replace('exec("del tensorflow")', 'pass # Patched for Python 3.13')
            with open(target_utils, "w", encoding="utf-8") as f:
                f.write(c)
            print("✅ Auto-patch Python 3.13 untuk OpenVINO berhasil diterapkan!")
except Exception as e:
    print(f"Info patch: {e}")

print(f"Versi OpenCV  : {cv2.__version__}")
print(f"Versi OpenVINO: {ov.__version__}")


✅ Auto-patch Python 3.13 untuk OpenVINO berhasil diterapkan!
Versi OpenCV  : 5.0.0
Versi OpenVINO: 2026.4.1-22982-e213a147257-releases/2026/4


### 📂 Langkah 2: Hubungkan Google Drive & Masukkan Path `known_faces`
Jalankan sel ini. Google Colab akan meminta izin menghubungkan Google Drive Anda.
Setelah terhubung, masukkan **Path folder `known_faces`** Anda pada kolom input di bawah.

*Contoh path umum di Google Drive:*
- `/content/drive/MyDrive/known_faces`
- `/content/drive/MyDrive/ESIS/AI Server/known_faces`
- `/content/drive/MyDrive/DATASET/WAJAH`

> 💡 **Tips mencari path di Colab:** Klik ikon folder (📁) di sidebar kiri Colab -> Buka folder `drive` -> `MyDrive` -> Cari folder `known_faces` -> Klik titik tiga -> Pilih **"Copy path"** -> Paste ke form di bawah.


In [ ]:
# @title 📁 Form Konfigurasi Path Google Drive { run: "auto" }

# 1. Hubungkan Google Drive
print("Sedang menghubungkan ke Google Drive Anda...")
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive berhasil terhubung!")

# 2. Masukkan Path ke folder dataset known_faces di Google Drive:
PATH_KNOWN_FACES_DRIVE = "/content/drive/MyDrive/Skripsi/AI/known_faces" # @param {type:"string"}

# 3. Tentukan folder di Google Drive untuk menyimpan hasil model (.h5, .xml, .bin, .npz)
PATH_OUTPUT_DRIVE = "/content/drive/MyDrive/Skripsi/OutputModel" # @param {type:"string"}

DIR_RAW = PATH_KNOWN_FACES_DRIVE
DIR_CROPPED = "/content/dataset_cropped_mtcnn"

os.makedirs(DIR_CROPPED, exist_ok=True)
os.makedirs(PATH_OUTPUT_DRIVE, exist_ok=True)

# 4. Validasi Keberadaan Folder Dataset
if not os.path.exists(DIR_RAW):
    print(f"❌ ERROR: Folder TIDAK DITEMUKAN di path: {DIR_RAW}")
    print("👉 Silakan periksa kembali penulisan path folder Anda di Google Drive.")
else:
    daftar_kelas = [d for d in os.listdir(DIR_RAW) if os.path.isdir(os.path.join(DIR_RAW, d)) and not d.startswith('.')]
    daftar_kelas.sort()

    total_foto = 0
    for k in daftar_kelas:
        sub = os.path.join(DIR_RAW, k)
        total_foto += len([f for f in os.listdir(sub) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    print("=" * 60)
    print("✅ DATASET GOOGLE DRIVE BERHASIL DIMUAT!")
    print("=" * 60)
    print(f"📍 Path Dataset Input  : {DIR_RAW}")
    print(f"💾 Path Simpan Hasil   : {PATH_OUTPUT_DRIVE}")
    print(f"👥 Total Mahasiswa     : {len(daftar_kelas)} orang")
    print(f"📸 Total Foto Wajah    : {total_foto} foto")
    print("=" * 60)
    print("Contoh 10 mahasiswa yang terdeteksi:")
    for k in daftar_kelas[:10]:
        print(f"  • {k}")
    if len(daftar_kelas) > 10:
        print(f"  ... dan {len(daftar_kelas) - 10} mahasiswa lainnya.")


Sedang menghubungkan ke Google Drive Anda...


### ✂️ Langkah 3: Deteksi & Pemotongan Wajah (MTCNN Preprocessing)


In [ ]:
print("Sedang memuat MTCNN Face Detector...")
detector = MTCNN()
print("✅ MTCNN siap digunakan! Memulai deteksi & cropping...")

sample_visualisasi = []
total_foto_diproses = 0
total_wajah_berhasil = 0

if os.path.exists(DIR_CROPPED):
    shutil.rmtree(DIR_CROPPED)
os.makedirs(DIR_CROPPED, exist_ok=True)

for kelas in tqdm(daftar_kelas, desc="Memproses Mahasiswa"):
    folder_sumber = os.path.join(DIR_RAW, kelas)
    folder_tujuan = os.path.join(DIR_CROPPED, kelas)
    os.makedirs(folder_tujuan, exist_ok=True)

    foto_files = [f for f in os.listdir(folder_sumber) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    ambil_sample = False

    for f in foto_files:
        total_foto_diproses += 1
        img_path = os.path.join(folder_sumber, f)

        try:
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            h, w = img_rgb.shape[:2]

            results = detector.detect_faces(img_rgb)
            if not results:
                continue

            best_face = max(results, key=lambda x: x['confidence'])
            if best_face['confidence'] < 0.85:
                continue

            x, y, fw, fh = best_face['box']
            m_x, m_y = int(fw * 0.1), int(fh * 0.1)
            x1 = max(0, x - m_x)
            y1 = max(0, y - m_y)
            x2 = min(w, x + fw + m_x)
            y2 = min(h, y + fh + m_y)

            face_crop = img_rgb[y1:y2, x1:x2]
            if face_crop.size == 0:
                continue

            face_pil = Img.fromarray(face_crop).resize((160, 160))
            out_file = os.path.join(folder_tujuan, f)
            face_pil.save(out_file)
            total_wajah_berhasil += 1

            if not ambil_sample:
                sample_visualisasi.append((np.asarray(face_pil), kelas))
                ambil_sample = True

        except Exception as e:
            pass

print(f"\n✅ Preprocessing MTCNN Selesai!")
print(f"Total foto diproses : {total_foto_diproses}")
print(f"Total wajah berhasil: {total_wajah_berhasil}")


### 🖼️ Langkah 4: Visualisasi Sampel Hasil Cropping Wajah


In [ ]:
if len(sample_visualisasi) > 0:
    n_samples = min(15, len(sample_visualisasi))
    cols = 5
    rows = (n_samples + cols - 1) // cols
    plt.figure(figsize=(15, 3 * rows))

    for i in range(n_samples):
        img, lbl = sample_visualisasi[i]
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        clean_lbl = lbl[:15] + ".." if len(lbl) > 15 else lbl
        plt.title(clean_lbl, fontsize=9)
        plt.axis('off')
    plt.suptitle("Sample Hasil Cropping MTCNN (160x160)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada sample foto yang berhasil di-crop.")


### 🧠 Langkah 5: Ekstraksi Fitur Biometrik dengan FaceNet


In [ ]:
print("Sedang memuat FaceNet Model...")
embedder = FaceNet()
print("✅ FaceNet Model Siap!")

X_embeddings = []
Y_labels = []

cropped_dirs = [d for d in os.listdir(DIR_CROPPED) if os.path.isdir(os.path.join(DIR_CROPPED, d))]
cropped_dirs.sort()

for kelas in tqdm(cropped_dirs, desc="Ekstraksi Fitur FaceNet"):
    folder_kelas = os.path.join(DIR_CROPPED, kelas)
    foto_files = [f for f in os.listdir(folder_kelas) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for f in foto_files:
        img_p = os.path.join(folder_kelas, f)
        try:
            face_img = Img.open(img_p).convert('RGB')
            face_arr = asarray(face_img)

            face_input = expand_dims(face_arr, axis=0)
            emb = embedder.embeddings(face_input)[0]
            emb = emb / np.linalg.norm(emb)

            X_embeddings.append(emb)
            Y_labels.append(kelas)
        except Exception as e:
            print(f"Error {f}: {e}")

X_embeddings = np.array(X_embeddings)
Y_labels = np.array(Y_labels)

FILE_NPZ = "data_wajah_facenet_mtcnn.npz"
np.savez_compressed(FILE_NPZ, x=X_embeddings, y=Y_labels)

print(f"\n✅ Database Wajah Berhasil Dibuat!")
print(f"File database           : {FILE_NPZ} ({os.path.getsize(FILE_NPZ) / 1024:.2f} KB)")
print(f"Total Vektor Embedding  : {X_embeddings.shape[0]}")
print(f"Dimensi Vektor Tiap Wajah: {X_embeddings.shape[1]}")


### 💾 Langkah 6: Export Model FaceNet ke Format Keras `.h5`


In [ ]:
FILE_H5 = "facenet_keras.h5"
print(f"Menyimpan model FaceNet ke format {FILE_H5}...")

keras_model = embedder.model
keras_model.save(FILE_H5)

size_mb = os.path.getsize(FILE_H5) / (1024 * 1024)
print(f"✅ Model Keras berhasil diekspor ke: {FILE_H5}")
print(f"Ukuran file: {size_mb:.2f} MB")


### ⚡ Langkah 7: Konversi & Optimasi Model ke Intel OpenVINO IR
*Dilengkapi auto-patch untuk kompatibilitas Python 3.13 PEP 667.*


In [ ]:
import os
import sys
import site
import subprocess

# ==============================================================================
# 1. CARI LOKASI OPENVINO TANPA MENG-IMPORT (Mencegah C-Extension Corrupt)
# ==============================================================================
site_dirs = site.getsitepackages() + [site.getusersitepackages()]
ov_dirs = []

for s in site_dirs:
    target = os.path.join(s, "openvino")
    if os.path.exists(target):
        ov_dirs.append(target)

# Patching file Python OpenVINO untuk kompatibilitas Python 3.13 (PEP 667)
patched_count = 0
for ov_root in ov_dirs:
    for root, _, files_list in os.walk(ov_root):
        for f in files_list:
            if f.endswith(".py"):
                p = os.path.join(root, f)
                try:
                    with open(p, "r", encoding="utf-8", errors="ignore") as fp:
                        content = fp.read()
                    if 'exec("del tensorflow")' in content:
                        content = content.replace('exec("del tensorflow")', 'pass # Patched for Python 3.13')
                        with open(p, "w", encoding="utf-8") as fp:
                            fp.write(content)
                        patched_count += 1
                        print(f"Patched: {os.path.basename(p)}")
                except Exception:
                    pass

print(f"Total file OpenVINO yang di-patch: {patched_count}")

# ==============================================================================
# 2. SEKARANG AMAN UNTUK IMPORT OPENVINO (Hanya 1x import bersih)
# ==============================================================================
import openvino as ov

print("Mengonversi model FaceNet Keras ke OpenVINO IR...")
ov_core = ov.Core()
OPENVINO_XML = "facenet_openvino.xml"

# ==============================================================================
# 3. PROSES KONVERSI MODEL (Dengan Fallback ONNX)
# ==============================================================================
ov_model = None
try:
    ov_model = ov.convert_model(keras_model, input=[(None, 160, 160, 3)])
    print("✅ Berhasil dikonversi langsung dari Keras model!")
except Exception as e_keras:
    print(f"Mengalihkan via ONNX Bridge karena kendala Python 3.13: {e_keras}")

    # Pastikan tf2onnx terinstall
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "tf2onnx"])
    import tf2onnx

    onnx_path = "facenet.onnx"
    print("Mengekspor model Keras ke format ONNX...")
    tf2onnx.convert.from_keras(keras_model, output_path=onnx_path)

    print("Mengonversi ONNX ke OpenVINO IR (.xml & .bin)...")
    ov_model = ov.convert_model(onnx_path)
    print("✅ Berhasil dikonversi via ONNX Bridge!")

# Simpan model OpenVINO
ov.save_model(ov_model, OPENVINO_XML)

size_xml = os.path.getsize("facenet_openvino.xml") / 1024
size_bin = os.path.getsize("facenet_openvino.bin") / (1024 * 1024)

print("=" * 50)
print("✅ KONVERSI OPENVINO BERHASIL 100%!")
print(f" - XML File : facenet_openvino.xml ({size_xml:.2f} KB)")
print(f" - BIN File : facenet_openvino.bin ({size_bin:.2f} MB)")
print("=" * 50)

### 🚀 Langkah 8: Kompilasi & Verifikasi OpenVINO Runtime di CPU


In [ ]:
compiled_model = ov_core.compile_model(
    model=ov_model,
    device_name="CPU",
    config={"PERFORMANCE_HINT": "LATENCY"}
)

input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)

print(f"Target Hardware Device: CPU ({ov_core.get_property('CPU', 'FULL_DEVICE_NAME')})")
print(f"Bentuk Input OpenVINO : {input_layer.shape}")
print(f"Bentuk Output OpenVINO: {output_layer.shape}")

dummy_input = np.random.uniform(0, 255, (1, 160, 160, 3)).astype(np.float32)

keras_out = keras_model(dummy_input, training=False).numpy()[0]
keras_out = keras_out / np.linalg.norm(keras_out)

ov_out = compiled_model([dummy_input])[output_layer][0]
ov_out = ov_out / np.linalg.norm(ov_out)

cosine_sim = np.dot(keras_out, ov_out)
print(f"\nVerifikasi Kemiripan Output Keras vs OpenVINO: {cosine_sim:.6f} / 1.000000")
if cosine_sim > 0.9999:
    print("✅ Presisi sempurna! Model OpenVINO 100% konsisten dengan Keras FaceNet.")


### 📊 Langkah 9: Benchmark Kecepatan (Keras CPU vs OpenVINO CPU)


In [ ]:
N_ITER = 50
test_face = np.random.uniform(0, 255, (1, 160, 160, 3)).astype(np.float32)

print(f"Menjalankan benchmark inferensi ({N_ITER} iterasi)...\n")

_ = keras_model(test_face, training=False)
start_k = time.time()
for _ in range(N_ITER):
    _ = keras_model(test_face, training=False)
durasi_k = time.time() - start_k
avg_ms_k = (durasi_k / N_ITER) * 1000
fps_k = N_ITER / durasi_k

_ = compiled_model([test_face])[output_layer]
start_ov = time.time()
for _ in range(N_ITER):
    _ = compiled_model([test_face])[output_layer]
durasi_ov = time.time() - start_ov
avg_ms_ov = (durasi_ov / N_ITER) * 1000
fps_ov = N_ITER / durasi_ov

speedup = avg_ms_k / avg_ms_ov

print("=" * 55)
print(f"{'Metrik':<20} | {'Keras CPU':<15} | {'OpenVINO CPU':<15}")
print("=" * 55)
print(f"{'Rata-rata Latensi':<20} | {avg_ms_k:.2f} ms{'':<8} | {avg_ms_ov:.2f} ms{'':<8}")
print(f"{'Throughput (FPS)':<20} | {fps_k:.2f} FPS{'':<7} | {fps_ov:.2f} FPS{'':<7}")
print(f"{'Peningkatan Efisiensi':<20} | 1.00x{'':<10} | {speedup:.2f}x Lebih Cepat!")
print("=" * 55)

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
plt.bar(['Keras', 'OpenVINO'], [avg_ms_k, avg_ms_ov], color=['#e74c3c', '#2ecc71'], width=0.5)
plt.title("Latensi Inferensi (ms) - Lebih Rendah Lebih Baik")
plt.ylabel("Milidetik (ms)")
for idx, v in enumerate([avg_ms_k, avg_ms_ov]):
    plt.text(idx, v + 1, f"{v:.1f} ms", ha='center', fontweight='bold')

plt.subplot(1, 2, 2)
plt.bar(['Keras', 'OpenVINO'], [fps_k, fps_ov], color=['#e74c3c', '#2ecc71'], width=0.5)
plt.title("Throughput Inferensi (FPS) - Lebih Tinggi Lebih Baik")
plt.ylabel("Frames Per Second (FPS)")
for idx, v in enumerate([fps_k, fps_ov]):
    plt.text(idx, v + 1, f"{v:.1f} FPS", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("benchmark_facenet_openvino.png", dpi=200)
plt.show()


### 🔍 Langkah 10: Uji Lengkap Pengenalan Wajah (MTCNN + OpenVINO Engine)


In [ ]:
def kenali_wajah(image_path, threshold=0.75):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("Gagal membaca gambar:", image_path)
        return

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    t_start = time.time()
    results = detector.detect_faces(img_rgb)
    t_detect = (time.time() - t_start) * 1000

    if not results:
        print("❌ Tidak ada wajah yang terdeteksi.")
        return

    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    ax = plt.gca()

    for face_info in results:
        if face_info['confidence'] < 0.80:
            continue

        x, y, fw, fh = face_info['box']
        m_x, m_y = int(fw * 0.1), int(fh * 0.1)
        x1, y1 = max(0, x - m_x), max(0, y - m_y)
        x2, y2 = min(w, x + fw + m_x), min(h, y + fh + m_y)

        crop = img_rgb[y1:y2, x1:x2]
        if crop.size == 0: continue

        face_pil = Img.fromarray(crop).resize((160, 160))
        face_input = expand_dims(asarray(face_pil).astype(np.float32), axis=0)

        t_emb_start = time.time()
        emb_target = compiled_model([face_input])[output_layer][0]
        emb_target = emb_target / np.linalg.norm(emb_target)
        t_emb = (time.time() - t_emb_start) * 1000

        dists = [np.linalg.norm(emb_target - db_emb) for db_emb in X_embeddings]
        min_dist = min(dists)
        best_idx = dists.index(min_dist)

        if min_dist < threshold:
            nama = Y_labels[best_idx]
            sim = max(0, (1 - (min_dist / threshold))) * 100
            label_text = f"{nama} ({sim:.1f}%)"
            box_color = 'green'
        else:
            label_text = f"Unknown ({min_dist:.2f})"
            box_color = 'red'

        rect = plt.Rectangle((x, y), fw, fh, fill=False, color=box_color, linewidth=2)
        ax.add_patch(rect)
        plt.text(x, y - 8, label_text, color='white', fontsize=10, fontweight='bold',
                 bbox=dict(facecolor=box_color, alpha=0.7, edgecolor='none'))

    plt.axis('off')
    plt.title(f"Deteksi MTCNN: {t_detect:.1f}ms | OpenVINO Embed: {t_emb:.1f}ms")
    plt.show()

sample_img = None
for root, _, f_list in os.walk(DIR_CROPPED):
    for fn in f_list:
        if fn.lower().endswith(('.jpg', '.png')):
            sample_img = os.path.join(root, fn)
            break
    if sample_img: break

if sample_img:
    print(f"Menguji sampel foto: {sample_img}")
    kenali_wajah(sample_img)


### 💾 Langkah 11: Simpan ke Google Drive & Download File Hasil Model


In [ ]:
ZIP_OUTPUT = "FaceNet_MTCNN_OpenVINO_Export.zip"
files_to_save = [
    "facenet_keras.h5",
    "facenet_openvino.xml",
    "facenet_openvino.bin",
    "data_wajah_facenet_mtcnn.npz",
    "benchmark_facenet_openvino.png"
]

print(f"📁 Menyimpan salinan model ke Google Drive: {PATH_OUTPUT_DRIVE}...")
for f in files_to_save:
    if os.path.exists(f):
        target_path = os.path.join(PATH_OUTPUT_DRIVE, f)
        shutil.copy2(f, target_path)
        print(f" ✅ Tersimpan di Drive: {target_path}")

print(f"\n📦 Mengemas model ke file zip {ZIP_OUTPUT}...")
with zipfile.ZipFile(ZIP_OUTPUT, 'w') as zf:
    for f in files_to_save:
        if os.path.exists(f):
            zf.write(f)

print(f"✅ File zip siap diunduh! ({os.path.getsize(ZIP_OUTPUT) / (1024*1024):.2f} MB)")
files.download(ZIP_OUTPUT)
